# MineDB Unified Self-Bootstrapping Pipeline (`Qwen/Qwen2.5-3B-Instruct` FP16 + MotherDuck)
Open-web multi-engine + Common Crawl CDX discovery, 1-hop listing spidering, strict salability & future-deadline validation, safe MotherDuck persistence, and automatic run summary save & session shutdown.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import asyncio
import hashlib
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from urllib.parse import parse_qs, unquote, urlparse, urlunparse

def log(msg: str):
    ts = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"[{ts} UTC] {msg}", flush=True)

log("=== PIPELINE STARTED ===")
log("[1.1] Initializing enhanced configuration & UAE-targeted query pools...")

# ----------------------------- CONFIG ---------------------------------------
DUCKDB_VERSION = "1.5.6"                  # must match MotherDuck-supported version
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"     # ~6.2GB FP16 weights; fits comfortably on 15GB T4
DATASET_SLUG = "minedb-cache-3b-v1"       # stored as <KAGGLE_USERNAME>/minedb-cache-3b-v1
DATASET_TITLE = "MineDB Cache 3B v1"

BUFFER_PATH = "/kaggle/input/raw-scrape-buffer/raw_batch.parquet"
CACHE_TMP_ROOT = "/tmp/minedb_cache"      # uses /tmp so /kaggle/working stays clean on save
MAX_PER_QUERY = 12
QUERIES_PER_NICHE_PER_RUN = 5             # rotate 5 queries/niche per 3h run to stay under SERP rate limits
MAX_CANDIDATE_URLS = 160                  # cap total candidate URLs per 3h cycle for predictable runtime
FETCH_CONCURRENCY = 12                    # balanced concurrency to avoid remote host 429s
RAW_TEXT_CHARS = 2200                     # extra context for richer contact/price/deadline extraction
LLM_BATCH_SIZE = 2                        # safe batch size on 15GB T4 (avoids 1.45GB logits spike)
MAX_NEW_TOKENS = 420                      # prevents JSON truncation on longer Arabic/multi-field outputs
MIN_VALUE_SCORE = 7
RETENTION_DAYS = 60
SEARCH_DELAY_S = 3.5                      # polite delay to avoid HTTP/2 & SERP rate-limiting
MAX_SEARCH_RETRIES = 2
AUTO_SAVE_AND_SHUTDOWN = True             # auto-save notebook snapshot & stop Kaggle session on completion

# Expanded UAE/Gulf-anchored query pools (rotated across runs to defeat seen_urls saturation)
NICHE_QUERY_POOL = {
    "real_estate": [
        "site:.ae Dubai Marina Downtown villa apartment sale price AED",
        "site:.ae Abu Dhabi Saadiyat Yas Island property sale AED",
        "site:.ae off plan handover payment plan Dubai Abu Dhabi",
        "site:.ae commercial office warehouse retail lease Dubai Abu Dhabi",
        "site:.ae distressed motivated seller property price drop Dubai",
        "site:propertyfinder.ae OR site:bayut.com OR site:dubizzle.com villa apartment AED",
        "Dubai Creek Harbour Business Bay penthouse townhouse sale AED",
        "Ras Al Khaimah Al Marjan Island beachfront property price AED",
        "Sharjah Aljada Masaar villa townhouse sale price AED",
        "DIFC ADGM Grade A commercial office space for rent AED",
    ],
    "b2b_leads": [
        "site:.ae careers hiring software cloud devops engineer Dubai Abu Dhabi",
        "DIFC OR ADGM OR Hub71 startup hiring AI data engineer",
        "site:gulftalent.com OR site:bayt.com OR site:naukrigulf.com cloud devops Dubai",
        "Dubai Internet City OR Masdar City company hiring cybersecurity architect",
        "UAE fintech enterprise SaaS hiring account executive Dubai Abu Dhabi",
        "site:.ae company careers engineering manager backend python kubernetes",
        "Abu Dhabi AI technology company hiring machine learning engineer",
        "Dubai logistics ecommerce enterprise hiring product manager data analyst",
        "UAE systems integrator cloud migration Microsoft AWS partner Dubai",
        "DIFC wealth management fintech hiring full stack developer",
    ],
    "ecommerce": [
        "site:.ae electronics laptop smartphone clearance sale price AED",
        "site:.ae myshopify OR collections flash sale discount AED",
        "Dubai Abu Dhabi appliance furniture outlet deal price AED",
        "site:noon.com OR site:amazon.ae OR site:sharafdg.com clearance deal AED",
        "UAE home kitchen coffee machine smart home bundle offer AED",
        "site:.ae luxury perfume oud watches online store offer AED",
        "Dubai gaming PC monitor GPU stock deal price AED",
        "UAE solar power station camping outdoor gear price AED",
        "site:.ae B2B wholesale office supplies packaging price AED",
        "Abu Dhabi Dubai fitness equipment treadmill sale price AED",
    ],
    "tenders": [
        "site:.ae tender RFP submission deadline procurement",
        "site:.gov.ae tender notice bid closing date",
        "Dubai Abu Dhabi municipality OR authority tender prequalification",
        "UAE oil gas energy construction tender invitation to bid",
        "DIFC OR ADGM OR DEWA OR ADNOC vendor registration RFP",
        "UAE IT cybersecurity software implementation RFP closing date",
        "Abu Dhabi Dubai healthcare hospital medical equipment tender",
        "UAE facilities management MEP maintenance tender notice",
        "Sharjah Ras Al Khaimah government procurement tender deadline",
        "UAE logistics supply chain consultancy request for proposal",
    ],
    "arabic_nlp": [
        "منتدى نقاش تجارب اسعار العقار دبي ابوظبي",
        "اسئلة واجوبة تقنية رواتب شركات الامارات",
        "تجربتي شراء عقار او استثمار في دبي",
        "رأيكم في السكن والعمل والمعيشة في ابوظبي ودبي",
        "مقارنة اسعار الاتصالات والخدمات والبنوك في الامارات",
        "نقاش حول المشاريع والتجارة الالكترونية في الخليج",
        "استفسار عن قوانين العمل والاقامة الذهبية الامارات",
        "افضل مناطق السكن للعائلات في دبي والشارقة",
        "تجارب الشراء من المتاجر الالكترونية في الامارات",
        "نقاش السوق العقاري في الامارات توقعات الاسعار",
    ],
}

# Select a deterministic-yet-rotating subset per 3-hour window so every run mines fresh queries
_time_slot = int(time.time() // (3 * 3600))
NICHE_QUERIES = {}
for _idx, (_niche, _pool) in enumerate(NICHE_QUERY_POOL.items()):
    _offset = (_time_slot + _idx) % len(_pool)
    _rotated = _pool[_offset:] + _pool[:_offset]
    NICHE_QUERIES[_niche] = _rotated[:QUERIES_PER_NICHE_PER_RUN]

NICHE_SCHEMA = {
    "real_estate": '{"price": "number or string or null", "currency": "AED or string or null", "location": "string or null", "property_type": "string or null", "urgency_signal": "string or null", "contact_info": "string or null"}',
    "b2b_leads":   '{"company_name": "string or null", "job_title": "string or null", "location": "UAE/Gulf city or null", "tech_stack": ["string"], "contact_or_url": "string or null"}',
    "ecommerce":   '{"product_name": "string or null", "brand": "string or null", "price": "string or number or null", "currency": "AED or string or null", "discount_pct": "string or number or null", "stock_status": "string or null"}',
    "tenders":     '{"tender_title": "string or null", "issuing_org": "string or null", "submission_deadline": "string or null", "sector": "string or null", "reference_or_contact": "string or null"}',
    "arabic_nlp":  '{"clean_arabic_text": "string or null", "dialect": "Gulf/Emirati/MSA or null", "domain_category": "string or null"}',
}

SYSTEM_PROMPT = (
    "You are a strict UAE & Gulf commercial data quality gatekeeper. Analyze the webpage text and output ONLY a single "
    "valid JSON object (no markdown fences, no commentary) with exact keys: "
    '"is_sellable" (boolean: true ONLY if the text contains a concrete, actionable UAE/Gulf commercial item—such as a '
    "specific UAE property listing for sale/rent with an actual numeric AED price and specific Dubai/Abu Dhabi/UAE community, "
    "an active UAE/Gulf job opening with a real company name and role, a concrete UAE product offer with numeric AED price, "
    "an active UAE/Gulf tender/RFP notice with issuing org or deadline, or genuine Arabic-script user discussion/Q&A; "
    "strictly FALSE for news articles, market reports, macro statistics like 'AED 72.6 billion' or percentage drops '-5.3%', "
    "generic landing pages without specific listings, 'no properties found' pages, court/eviction news, or non-UAE pages), "
    '"value_score" (integer 1-10 rating commercial specificity and completeness; >=7 requires a concrete listing/offer/tender/job '
    "with >=2 concrete non-null fields from the target schema), "
    '"extracted_data" (JSON object strictly matching the user schema; price MUST be a scalar numeric/AED price string like '
    "'AED 1,850,000' or 'AED 95,000/yr'—never a list, dict, percentage, building name, or macro market figure; "
    "use JSON null for missing fields—never output placeholder strings like 'Unknown', 'N/A', 'None', or hallucinated facts)."
)

log(f"  -> Target Model: {MODEL_ID} | Cache Slug: {DATASET_SLUG} | Slot: {_time_slot} ({QUERIES_PER_NICHE_PER_RUN} queries/niche)")
log(f"  -> Batch Size: {LLM_BATCH_SIZE} | Max Tokens: {MAX_NEW_TOKENS} | Min Score: {MIN_VALUE_SCORE} | Auto-Save: {AUTO_SAVE_AND_SHUTDOWN}")

# ----------------------------- SECRETS & SELF-BOOTSTRAPPING CACHE -----------
log("[1.2] Loading credentials (env vars / secrets.json / Kaggle Secrets)...")
FILE_SECRETS = {}
for root, _, files in os.walk("/kaggle/input"):
    if "secrets.json" in files:
        secret_path = os.path.join(root, "secrets.json")
        with open(secret_path, "r", encoding="utf-8") as f:
            FILE_SECRETS = json.load(f)
        log(f"  -> Found mounted secrets file: {secret_path}")
        break


def get_secret_safe(name, required=True):
    val = os.environ.get(name) or FILE_SECRETS.get(name)
    source = "env/file"
    if not val:
        try:
            from kaggle_secrets import UserSecretsClient
            val = UserSecretsClient().get_secret(name)
            source = "kaggle_secrets"
        except Exception:
            val = None
    if val:
        log(f"  -> Secret '{name}' loaded OK (source={source}, len={len(val.strip())})")
        return val.strip().strip('"').strip("'")
    if required:
        raise ValueError(f"Missing required secret: '{name}'.")
    log(f"  -> Optional secret '{name}' not set.")
    return None


md_token = get_secret_safe("MOTHERDUCK_TOKEN", required=True)
if not md_token.startswith("eyJ"):
    raise ValueError(
        "MOTHERDUCK_TOKEN does not look like a JWT (expected to start with 'eyJ'). "
        "Check GitHub Actions Secret MOTHERDUCK_TOKEN."
    )
os.environ["MOTHERDUCK_TOKEN"] = md_token

kaggle_user = get_secret_safe("KAGGLE_USERNAME", required=True)
os.environ["KAGGLE_USERNAME"] = kaggle_user

kaggle_api_token = get_secret_safe("KAGGLE_API_TOKEN", required=False)
if kaggle_api_token:
    os.environ["KAGGLE_API_TOKEN"] = kaggle_api_token
else:
    os.environ["KAGGLE_KEY"] = get_secret_safe("KAGGLE_KEY", required=True)

DATASET_ID = f"{kaggle_user}/{DATASET_SLUG}"

REQUIRED_PKGS = [
    f"duckdb=={DUCKDB_VERSION}",
    "httpx",
    "trafilatura",
    "ddgs",
    "accelerate",
]


def is_valid_wheel_cache(wheels_dir: str) -> bool:
    if not wheels_dir or not os.path.isdir(wheels_dir):
        return False
    files = [f.lower() for f in os.listdir(wheels_dir) if f.endswith(".whl")]
    return any(f.startswith(f"duckdb-{DUCKDB_VERSION}") for f in files)


def is_valid_model_cache(model_dir: str) -> bool:
    if not model_dir or not os.path.isdir(model_dir):
        return False
    files = os.listdir(model_dir)
    return "config.json" in files and any(f.endswith(".safetensors") for f in files)


def find_existing_cache():
    for root_dir in ("/kaggle/input", CACHE_TMP_ROOT):
        if not os.path.exists(root_dir):
            continue
        wheels_dir, model_dir = None, None
        for r, dirs, _ in os.walk(root_dir):
            if "wheels" in dirs and wheels_dir is None:
                cand = os.path.join(r, "wheels")
                if is_valid_wheel_cache(cand):
                    wheels_dir = cand
            if "model" in dirs and model_dir is None:
                cand = os.path.join(r, "model")
                if is_valid_model_cache(cand):
                    model_dir = cand
            if wheels_dir and model_dir:
                return wheels_dir, model_dir
    return None, None


def ensure_offline_cache():
    log("[1.3] Checking local disk (/kaggle/input & /tmp/minedb_cache) for existing cache...")
    wheels_dir, model_dir = find_existing_cache()
    if wheels_dir and model_dir:
        whl_count = len([f for f in os.listdir(wheels_dir) if f.endswith(".whl")])
        shard_count = len([f for f in os.listdir(model_dir) if f.endswith(".safetensors")])
        log(f"  -> CACHE HIT! Using local cache: wheels={wheels_dir} ({whl_count} .whl) | model={model_dir} ({shard_count} shards)")
        return wheels_dir, model_dir

    log(f"  -> No local cache found. Checking remote Kaggle Dataset '{DATASET_ID}'...")
    status = subprocess.run(
        ["kaggle", "datasets", "status", DATASET_ID],
        capture_output=True,
        text=True,
    )
    if status.returncode == 0:
        status_line = status.stdout.strip().splitlines()[-1] if status.stdout.strip() else "ready"
        log(f"  -> Remote dataset '{DATASET_ID}' exists (status: {status_line}). Attempting download to {CACHE_TMP_ROOT}...")
        t0 = time.time()
        shutil.rmtree(CACHE_TMP_ROOT, ignore_errors=True)
        os.makedirs(CACHE_TMP_ROOT, exist_ok=True)
        dl = subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET_ID, "-p", CACHE_TMP_ROOT, "--unzip", "-q"],
            capture_output=True,
            text=True,
        )
        if dl.returncode == 0:
            wheels_dir, model_dir = find_existing_cache()
            if wheels_dir and model_dir:
                log(f"  -> Downloaded & verified cache in {time.time() - t0:.1f}s.")
                return wheels_dir, model_dir
            log("  -> Dataset downloaded but valid wheels/model folders were missing; rebuilding cache...")
        else:
            err_msg = (dl.stderr or dl.stdout).strip().splitlines()[-1] if (dl.stderr or dl.stdout).strip() else "unknown error"
            log(f"  -> Remote dataset download failed ({err_msg}) — dataset shell is empty/incomplete; rebuilding & re-publishing...")

    log(f"  -> Remote dataset not found. Starting first-time bootstrap for '{DATASET_ID}'...")
    shutil.rmtree(CACHE_TMP_ROOT, ignore_errors=True)
    wheels_dir = os.path.join(CACHE_TMP_ROOT, "wheels")
    model_dir = os.path.join(CACHE_TMP_ROOT, "model")
    os.makedirs(wheels_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    t0 = time.time()
    log(f"  -> [Bootstrap 1/3] Downloading binary wheels: {REQUIRED_PKGS}...")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "download", "-q",
            "--only-binary=:all:",
            "--dest", wheels_dir,
            *REQUIRED_PKGS,
        ],
        timeout=300,
    )
    log(f"  -> [Bootstrap 1/3] Downloaded {len(os.listdir(wheels_dir))} wheel files in {time.time() - t0:.1f}s.")

    t1 = time.time()
    log(f"  -> [Bootstrap 2/3] Downloading HuggingFace model snapshot '{MODEL_ID}'...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=model_dir,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot", "*.tflite", "*.bin"],
    )
    shutil.rmtree(os.path.join(model_dir, ".cache"), ignore_errors=True)
    log(f"  -> [Bootstrap 2/3] Model snapshot saved to {model_dir} in {time.time() - t1:.1f}s.")

    t2 = time.time()
    log(f"  -> [Bootstrap 3/3] Publishing '{DATASET_ID}' to Kaggle Datasets...")
    meta = {
        "title": DATASET_TITLE,
        "id": DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(CACHE_TMP_ROOT, "dataset-metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    res = subprocess.run(
        ["kaggle", "datasets", "create", "-p", CACHE_TMP_ROOT, "--dir-mode", "zip"],
        capture_output=True,
        text=True,
    )
    if res.stdout.strip():
        log(f"     kaggle stdout: {res.stdout.strip()}")
    if res.stderr.strip():
        log(f"     kaggle stderr: {res.stderr.strip()}")
    if "Dataset creation error" in (res.stdout + res.stderr) or res.returncode != 0:
        log("  -> Dataset already exists or create returned non-zero; pushing new dataset version...")
        subprocess.run(
            ["kaggle", "datasets", "version", "-p", CACHE_TMP_ROOT, "-m", "Update cache", "--dir-mode", "zip"],
            check=False,
        )
    log(f"  -> [Bootstrap 3/3] Publish finished in {time.time() - t2:.1f}s.")
    return wheels_dir, model_dir


WHEELS_DIR, CACHED_MODEL_DIR = ensure_offline_cache()

log("[1.4] Installing Python packages offline from wheel cache...")
t_pip = time.time()
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-index",
        f"--find-links={WHEELS_DIR}", "--force-reinstall", "--no-deps",
        f"duckdb=={DUCKDB_VERSION}",
    ],
    timeout=120,
)
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-index",
        f"--find-links={WHEELS_DIR}",
        *[p for p in REQUIRED_PKGS if not p.startswith("duckdb==")],
    ],
    timeout=240,
)
log(f"  -> Offline pip install complete in {time.time() - t_pip:.1f}s.")

import duckdb
import httpx
import pandas as pd
import torch
import trafilatura
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS

log(f"[1.5] Runtime verified: DuckDB v{duckdb.__version__} | PyTorch {torch.__version__} | CUDA GPUs: {torch.cuda.device_count()}")
if duckdb.__version__ != DUCKDB_VERSION:
    raise RuntimeError(
        f"DuckDB v{duckdb.__version__} was already loaded in memory; click 'Run -> Restart & Clear Cell Outputs' and re-run."
    )


def run_async(coro):
    """Works both in plain scripts and inside Kaggle/Jupyter (running event loop)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(asyncio.run, coro).result()


def save_and_shutdown_session():
    """Flush outputs, save a run summary artifact to /kaggle/working, and stop the Kaggle session cleanly."""
    if not AUTO_SAVE_AND_SHUTDOWN:
        log("[7.0] AUTO_SAVE_AND_SHUTDOWN is False; leaving session running.")
        return

    log("[7.1] Saving run summary & notebook snapshot before session shutdown...")
    try:
        os.makedirs("/kaggle/working", exist_ok=True)
        summary_path = "/kaggle/working/last_run_summary.json"
        summary = {
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "model_id": MODEL_ID,
            "dataset_id": DATASET_ID,
            "time_slot": _time_slot,
            "min_value_score": MIN_VALUE_SCORE,
        }
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        log(f"  -> Wrote run summary to {summary_path}")

        kernel_slug = os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
        log(f"  -> Kernel environment detected (KAGGLE_KERNEL_RUN_TYPE={kernel_slug}).")
    except Exception as e:
        log(f"  -> Non-fatal error during save prep: {e}")

    log("[7.2] Shutting down Kaggle session now...")
    time.sleep(1.5)
    sys.stdout.flush()
    sys.stderr.flush()
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None and getattr(ip, "kernel", None) is not None:
            ip.kernel.do_shutdown(restart=False)
    except Exception:
        pass
    # Give MotherDuck background sync threads a moment to flush before exiting
    time.sleep(2.5)
    os._exit(0)

In [ ]:
log(f"[2.1] Connecting to MotherDuck database 'md:minedb' (DuckDB v{duckdb.__version__})...")
t_db = time.time()
con = duckdb.connect("md:minedb")
log(f"  -> Connected to MotherDuck in {time.time() - t_db:.2f}s.")

log("[2.2] Ensuring 'seen_urls' and niche tables exist...")
con.execute("""
    CREATE TABLE IF NOT EXISTS seen_urls (
        source_url VARCHAR PRIMARY KEY,
        first_seen TIMESTAMPTZ
    )
""")
seen_total = con.execute("SELECT COUNT(*) FROM seen_urls").fetchone()[0]
log(f"  -> Table 'seen_urls' ready (currently tracking {seen_total} historical URLs).")

for _niche in NICHE_QUERIES:
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS {_niche} (
            niche VARCHAR, source_url VARCHAR, scraped_at TIMESTAMPTZ,
            value_score INTEGER, cleaned_json VARCHAR
        )
    """)
    row_cnt = con.execute(f"SELECT COUNT(*) FROM {_niche}").fetchone()[0]
    log(f"  -> Table 'minedb.{_niche}' ready ({row_cnt} existing rows).")

In [ ]:
def get_con():
    global con
    try:
        con.execute("SELECT 1")
    except Exception:
        log("  -> MotherDuck connection dropped; reconnecting...")
        con = duckdb.connect("md:minedb")
    return con


# ----------------------------- URL NORMALIZATION & OPEN-WEB DISCOVERY -------
TRACKING_PARAMS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "fbclid", "gclid", "mc_cid", "mc_eid", "ref", "source", "msockid",
}

SKIP_EXTENSIONS = (
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".jpg", ".jpeg", ".png", ".gif", ".webp", ".svg", ".mp4", ".zip",
    ".xml", ".gz", ".json", ".css", ".js",
)

NON_UAE_MARKERS = (
    "spokane", "-wa", "seattle", "san francisco", "new york", "itjobswatch",
    "dev-korea", "seoul", "subito.it", "ikea.com/ch/it", "ikea.com/it", "hl=zh-hant",
    "globalremotely.com", "wantremote.com", "fastaijobs.com", "fundraiseinsider.com",
    "linkedin.com/jobs/hiring-jobs-", ".co.uk/", ".gov.uk/", ".ca/", ".au/",
)

# Generic news/editorial/aggregator domains that produce macro stats instead of actionable listings
NOISE_DOMAINS = (
    "gulfnews.com", "khaleejtimes.com", "arabianbusiness.com", "zawya.com",
    "gulfbusiness.com", "propertynews.ae", "bazaartimes.com", "wamda.com",
    "magnitt.com", "panicselling.com", "aurantius.ae", "continentalclub.ae",
    "wikipedia.org", "youtube.com", "facebook.com", "instagram.com", "tiktok.com",
)

UAE_RELEVANCE_TERMS = (
    "uae", "dubai", "abu dhabi", "abudhabi", "sharjah", "ajman", "ras al khaimah",
    "fujairah", "umm al quwain", "emirates", "difc", "adgm", "hub71", "masdar",
    "aed", "dhs", "dirham", ".ae", "gulf", "khaleej", "الامارات", "الإمارات",
    "دبي", "ابوظبي", "أبوظبي", "الشارقة", "الخليج",
)

# Common Crawl CDX wildcard URL patterns across the entire .ae & UAE commercial web (no hardcoded site list)
COMMON_CRAWL_PATTERNS = {
    "real_estate": [
        "*.ae/*for-sale*",
        "*.ae/*villa*dubai*",
        "*.ae/*apartment*dubai*",
        "*.ae/*property*abu-dhabi*",
        "*.ae/*distress*",
    ],
    "b2b_leads": [
        "*.ae/*careers*",
        "*.ae/*jobs*",
        "*.ae/*hiring*",
        "*.ae/*engineer*",
    ],
    "ecommerce": [
        "*.ae/*products/*",
        "*.ae/*collections/*",
        "*.ae/*shop/*",
        "*.ae/*deal*",
    ],
    "tenders": [
        "*.ae/*tender*",
        "*.ae/*rfp*",
        "*.ae/*procurement*",
        "*.gov.ae/*tender*",
    ],
    "arabic_nlp": [
        "*.ae/*forum*",
        "*.ae/*montada*",
        "*.ae/*question*",
        "*.ae/*ar/*",
    ],
}

SEARXNG_INSTANCES = [
    "https://searx.be/search",
    "https://search.sapti.me/search",
    "https://priv.au/search",
    "https://searx.tiekoetter.com/search",
]


def is_off_region_or_noise_url(url: str, niche: str) -> bool:
    u = (url or "").lower()
    if any(d in u for d in NOISE_DOMAINS):
        return True
    if niche == "arabic_nlp":
        return False
    return any(m in u for m in NON_UAE_MARKERS)


def has_uae_signal(url: str, text: str, niche: str) -> bool:
    """Ensure scraped page actually contains UAE/Gulf or Arabic signals before spending GPU cycles."""
    combined = f"{url} {text}".lower()
    if niche == "arabic_nlp":
        return len(re.findall(r"[\u0600-\u06FF]", text or "")) >= 25
    return any(term in combined for term in UAE_RELEVANCE_TERMS)


def normalize_url(raw_url: str) -> str:
    """Canonicalize URLs (strip tracking params & fragments) to maximize deduplication."""
    if not raw_url or not raw_url.startswith(("http://", "https://")):
        return ""
    try:
        p = urlparse(raw_url.strip())
        if any(p.path.lower().endswith(ext) for ext in SKIP_EXTENSIONS):
            return ""
        qs = parse_qs(p.query, keep_blank_values=False)
        clean_qs = "&".join(
            f"{k}={v[0]}" for k, v in sorted(qs.items()) if k.lower() not in TRACKING_PARAMS
        )
        clean_path = p.path.rstrip("/") or "/"
        return urlunparse((p.scheme.lower(), p.netloc.lower(), clean_path, "", clean_qs, ""))
    except Exception:
        return ""


USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0",
]

SEARCH_BACKENDS = ["bing", "brave", "mojeek", "yahoo", "duckduckgo"]


def _extract_ddg_links(html_text: str, max_results: int) -> list[str]:
    urls = []
    for raw_href in re.findall(r'class="(?:result__a|result-link)"[^>]*href="([^"]+)"', html_text):
        if "uddg=" in raw_href:
            qs = parse_qs(urlparse(raw_href).query)
            target = unquote(qs.get("uddg", [""])[0])
        else:
            target = raw_href
        if target.startswith("http") and "duckduckgo.com" not in target and target not in urls:
            urls.append(target)
            if len(urls) >= max_results:
                break
    return urls


def ddg_html_fallback(query: str, niche: str, max_results: int = 12) -> list[str]:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "ar-AE,ar;q=0.9,en-AE;q=0.8,en;q=0.7",
        "Referer": "https://html.duckduckgo.com/",
    }
    region_code = "xa-ar" if niche == "arabic_nlp" else "ae-en"
    try:
        with httpx.Client(headers=headers, http2=False, follow_redirects=True, timeout=7.0) as client:
            resp = client.post(
                "https://html.duckduckgo.com/html/",
                data={"q": query, "kl": region_code},
            )
            if resp.status_code == 200 and "anomaly-modal" not in resp.text:
                return _extract_ddg_links(resp.text, max_results)
    except Exception:
        pass
    return []


def searxng_search(query: str, niche: str, max_results: int = 12) -> list[str]:
    """Query open federated SearXNG JSON endpoints across the open web when DDGS throttles."""
    lang = "ar" if niche == "arabic_nlp" else "en-AE"
    headers = {"User-Agent": random.choice(USER_AGENTS), "Accept": "application/json"}
    instances = SEARXNG_INSTANCES[:]
    random.shuffle(instances)
    with httpx.Client(headers=headers, http2=False, follow_redirects=True, timeout=7.0) as client:
        for base_url in instances:
            try:
                resp = client.get(
                    base_url,
                    params={"q": query, "format": "json", "language": lang},
                )
                if resp.status_code == 200:
                    data = resp.json()
                    urls = [r["url"] for r in data.get("results", []) if r.get("url")]
                    if urls:
                        return urls[:max_results]
            except Exception:
                continue
    return []


def _query_ddgs_bounded(ddgs_obj, query: str, backend: str, region: str, max_results: int) -> list[str]:
    with ThreadPoolExecutor(max_workers=1) as ex:
        fut = ex.submit(
            lambda: list(
                ddgs_obj.text(
                    query,
                    region=region,
                    backend=backend,
                    max_results=max_results,
                ) or []
            )
        )
        try:
            res = fut.result(timeout=6.5)
            return [r.get("href") or r.get("url") for r in res if (r.get("href") or r.get("url"))]
        except Exception:
            return []


def harvest_common_crawl_open_web(add_url_fn) -> int:
    """Query Common Crawl's open web index (CDX API) for fresh *.ae commercial pages across all 5 niches."""
    log("[3.1b] Querying Common Crawl CDX open-web index for fresh UAE (*.ae) pages...")
    total_added = 0
    headers = {"User-Agent": random.choice(USER_AGENTS)}
    # Latest 2026 Common Crawl indexes (November 2026 CC-MAIN-2026-47 + October 2026 fallback)
    cdx_endpoints = [
        "https://index.commoncrawl.org/CC-MAIN-2026-47-index",
        "https://index.commoncrawl.org/CC-MAIN-2026-43-index",
    ]
    with httpx.Client(headers=headers, http2=False, follow_redirects=True, timeout=12.0) as client:
        for niche, patterns in COMMON_CRAWL_PATTERNS.items():
            pattern = patterns[_time_slot % len(patterns)]
            resp = None
            for cdx_endpoint in cdx_endpoints:
                try:
                    r = client.get(
                        cdx_endpoint,
                        params={
                            "url": pattern,
                            "output": "json",
                            "filter": "status:200",
                            "limit": 35,
                        },
                    )
                    if r.status_code == 200 and r.text.strip():
                        resp = r
                        break
                except Exception:
                    continue
            if resp is None:
                continue
            try:
                if resp.status_code != 200:
                    continue
                added = 0
                lines = [ln for ln in resp.text.splitlines() if ln.strip()]
                random.shuffle(lines)
                for ln in lines:
                    try:
                        rec = json.loads(ln)
                        u = rec.get("url", "")
                        if add_url_fn(niche, u):
                            added += 1
                            total_added += 1
                            if added >= 15:
                                break
                    except Exception:
                        continue
            except Exception:
                continue
    log(f"  -> Common Crawl open-web index added {total_added} candidate URLs across niches.")
    return total_added


def discover_urls():
    log("[3.1] Running open-web multi-engine discovery across 5 UAE niches...")
    found, seen = [], set()
    total_queries = sum(len(qs) for qs in NICHE_QUERIES.values())
    consecutive_empty = 0
    serp_throttled = False

    def add_url(niche: str, raw_url: str) -> bool:
        norm = normalize_url(raw_url)
        if norm and norm not in seen and not is_off_region_or_noise_url(norm, niche):
            seen.add(norm)
            found.append((niche, norm))
            return True
        return False

    with DDGS(timeout=6) as ddgs:
        q_idx = 0
        for niche, queries in NICHE_QUERIES.items():
            for q in queries:
                q_idx += 1
                hits = []
                used_backend = "none"
                region = "xa-ar" if niche == "arabic_nlp" else "ae-en"

                if not serp_throttled:
                    ordered_backends = (
                        SEARCH_BACKENDS[q_idx % len(SEARCH_BACKENDS):]
                        + SEARCH_BACKENDS[:q_idx % len(SEARCH_BACKENDS)]
                    )
                    for backend in ordered_backends:
                        hits = _query_ddgs_bounded(ddgs, q, backend=backend, region=region, max_results=MAX_PER_QUERY)
                        if hits:
                            used_backend = backend
                            break

                    if not hits:
                        hits = searxng_search(q, niche=niche, max_results=MAX_PER_QUERY)
                        if hits:
                            used_backend = "searxng"

                    if not hits:
                        hits = ddg_html_fallback(q, niche=niche, max_results=MAX_PER_QUERY)
                        if hits:
                            used_backend = "ddg-html"

                if hits:
                    consecutive_empty = 0
                else:
                    consecutive_empty += 1

                added = sum(1 for href in hits if add_url(niche, href))
                log(f"  -> Query [{q_idx}/{total_queries}] [{niche}] '{q}' -> backend={used_backend}, raw_hits={len(hits)}, unique_added={added}")

                if consecutive_empty >= 6 and not serp_throttled:
                    serp_throttled = True
                    log("  -> Live SERP engines throttled on runner IP; shifting to Common Crawl open-web index + link spidering.")
                    break

                if not serp_throttled:
                    time.sleep(max(1.2, SEARCH_DELAY_S * 0.4) + random.uniform(0.2, 0.6))

            if serp_throttled:
                break

    # Supplement with open-web Common Crawl CDX discovery whenever SERPs yield < 60 URLs
    if len(found) < 60:
        harvest_common_crawl_open_web(add_url)

    if len(found) > MAX_CANDIDATE_URLS:
        random.shuffle(found)
        found = found[:MAX_CANDIDATE_URLS]
        log(f"  -> Capped candidate URLs to {MAX_CANDIDATE_URLS} for runtime stability.")
    log(f"  -> Discovery complete: {len(found)} unique open-web candidate URLs collected.")
    return found


def filter_unseen(pairs):
    log("[3.2] Filtering candidate URLs against MotherDuck 'seen_urls'...")
    db = get_con()
    existing = {r[0] for r in db.execute("SELECT source_url FROM seen_urls").fetchall()}
    unseen = [(n, u) for n, u in pairs if u not in existing]
    log(f"  -> {len(pairs)} candidates -> {len(pairs) - len(unseen)} already seen -> {len(unseen)} brand-new URLs to scrape.")
    return unseen


# ----------------------------- SCRAPING + 1-HOP LINK EXPANSION --------------
def extract_same_domain_listing_links(base_url: str, html_text: str, niche: str, max_links: int = 4) -> list[str]:
    """When a search/index page is fetched, follow deep listing/detail links on the same domain."""
    parsed_base = urlparse(base_url)
    base_host = parsed_base.netloc.lower()
    keywords = {
        "real_estate": ("villa", "apartment", "penthouse", "townhouse", "property", "sale", "rent", "aed"),
        "b2b_leads": ("job", "career", "hiring", "engineer", "developer", "manager", "vacancy", "role"),
        "ecommerce": ("product", "item", "shop", "buy", "deal", "offer", "aed"),
        "tenders": ("tender", "rfp", "bid", "procurement", "notice", "project"),
        "arabic_nlp": ("thread", "topic", "question", "forum", "post", "ar"),
    }.get(niche, ())
    discovered = []
    for href in re.findall(r'href=["\']([^"\'#]+)["\']', html_text, flags=re.IGNORECASE):
        if href.startswith("/"):
            cand = f"{parsed_base.scheme}://{base_host}{href}"
        elif href.startswith("http"):
            cand = href
        else:
            continue
        norm = normalize_url(cand)
        if not norm or norm == base_url:
            continue
        p = urlparse(norm)
        if p.netloc.lower() != base_host or len(p.path.strip("/").split("/")) < 2:
            continue
        if any(k in norm.lower() for k in keywords) and norm not in discovered:
            discovered.append(norm)
            if len(discovered) >= max_links:
                break
    return discovered


async def fetch_and_extract(client, sem, niche, url, stats, discovered_sublinks):
    async with sem:
        for attempt in range(2):
            try:
                headers = {
                    "User-Agent": random.choice(USER_AGENTS),
                    "Accept-Language": "en-AE,en;q=0.9,ar-AE;q=0.8",
                }
                resp = await client.get(url, headers=headers, timeout=14.0)
                if resp.status_code in (429, 502, 503, 504) and attempt == 0:
                    await asyncio.sleep(1.2)
                    continue
                if resp.status_code >= 400:
                    stats["http_err"] += 1
                    return None
                ctype = resp.headers.get("content-type", "").lower()
                if "html" not in ctype and "text" not in ctype:
                    stats["non_html"] += 1
                    return None
                # Harvest deeper same-domain listing URLs from category/search pages
                for sub_u in extract_same_domain_listing_links(url, resp.text, niche, max_links=3):
                    discovered_sublinks.append((niche, sub_u))

                text = await asyncio.to_thread(
                    trafilatura.extract,
                    resp.text,
                    include_comments=False,
                    include_tables=True,
                    favor_precision=True,
                )
                if text:
                    clean_text = re.sub(r"\s+", " ", text).strip()
                    if len(clean_text) > 180:
                        if not has_uae_signal(url, clean_text, niche):
                            stats["non_uae"] += 1
                            return None
                        stats["ok"] += 1
                        return {
                            "niche": niche,
                            "source_url": url,
                            "raw_text": clean_text[:RAW_TEXT_CHARS],
                            "scraped_at": datetime.now(timezone.utc),
                        }
                stats["short_or_empty"] += 1
                return None
            except Exception:
                if attempt == 0:
                    await asyncio.sleep(0.7)
        stats["timeout_or_conn"] += 1
    return None


async def scrape(targets):
    log(f"[3.3] Scraping & extracting text from {len(targets)} URLs + 1-hop listing spidering (concurrency={FETCH_CONCURRENCY})...")
    t_scrape = time.time()
    stats = {"ok": 0, "http_err": 0, "non_html": 0, "short_or_empty": 0, "timeout_or_conn": 0, "non_uae": 0}
    discovered_sublinks = []
    sem = asyncio.Semaphore(FETCH_CONCURRENCY)
    limits = httpx.Limits(max_connections=FETCH_CONCURRENCY * 2, max_keepalive_connections=FETCH_CONCURRENCY)
    async with httpx.AsyncClient(follow_redirects=True, limits=limits) as client:
        results = await asyncio.gather(*(fetch_and_extract(client, sem, n, u, stats, discovered_sublinks) for n, u in targets))
        valid = [r for r in results if r]

        # 1-hop follow-up crawl on discovered deep listing pages (up to 45 unseen sub-URLs)
        db_local = get_con()
        existing_seen = {r[0] for r in db_local.execute("SELECT source_url FROM seen_urls").fetchall()}
        already_tried = {u for _, u in targets}
        hop2_targets = []
        hop2_seen = set()
        for n, su in discovered_sublinks:
            if su not in existing_seen and su not in already_tried and su not in hop2_seen and not is_off_region_or_noise_url(su, n):
                hop2_seen.add(su)
                hop2_targets.append((n, su))
                if len(hop2_targets) >= 45:
                    break
        if hop2_targets:
            log(f"  -> Following {len(hop2_targets)} deep listing links discovered during initial crawl...")
            hop2_sublinks = []
            hop2_results = await asyncio.gather(*(fetch_and_extract(client, sem, n, u, stats, hop2_sublinks) for n, u in hop2_targets))
            valid.extend([r for r in hop2_results if r])
            targets.extend(hop2_targets)

    log(
        f"  -> Scrape finished in {time.time() - t_scrape:.1f}s | "
        f"valid={stats['ok']}, non_uae_dropped={stats['non_uae']}, http_err={stats['http_err']}, "
        f"short/empty={stats['short_or_empty']}, timeout/conn={stats['timeout_or_conn']}, non_html={stats['non_html']}"
    )
    if valid:
        niche_counts = pd.Series([r["niche"] for r in valid]).value_counts().to_dict()
        log(f"  -> Extracted pages per niche: {niche_counts}")
    return pd.DataFrame(valid)


db = get_con()
if os.path.exists(BUFFER_PATH):
    log(f"[3.1] Found parquet buffer at {BUFFER_PATH}; loading unseen rows...")
    df = pd.read_parquet(BUFFER_PATH)
    df["source_url"] = df["source_url"].map(normalize_url)
    df = df[df["source_url"] != ""]
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], utc=True)
    df = df.drop_duplicates("source_url")
    seen_set = {u for (u,) in db.execute("SELECT source_url FROM seen_urls").fetchall()}
    df = df[~df["source_url"].isin(seen_set)]
    attempted = df["source_url"].tolist()
    log(f"  -> Loaded {len(df)} unseen rows from parquet buffer.")
else:
    targets = filter_unseen(discover_urls())
    df = run_async(scrape(targets)) if targets else pd.DataFrame()
    attempted = [u for _, u in targets]


def mark_seen(urls):
    if not urls:
        return
    db = get_con()
    now = datetime.now(timezone.utc)
    unique_urls = list(dict.fromkeys(urls))
    seen_df = pd.DataFrame({"source_url": unique_urls, "first_seen": now})
    try:
        db.unregister("seen_df_view")
    except Exception:
        pass
    db.register("seen_df_view", seen_df)
    db.execute(
        "INSERT OR IGNORE INTO seen_urls (source_url, first_seen) "
        "SELECT source_url, first_seen FROM seen_df_view"
    )
    try:
        db.unregister("seen_df_view")
    except Exception:
        pass
    log(f"  -> Logged {len(unique_urls)} attempted URLs into 'minedb.seen_urls'.")


def apply_retention():
    db = get_con()
    cutoff = f"CAST(NOW() AS TIMESTAMPTZ) - INTERVAL '{RETENTION_DAYS} days'"
    db.execute(f"DELETE FROM seen_urls WHERE first_seen < {cutoff}")
    for n in NICHE_QUERIES:
        db.execute(f"DELETE FROM {n} WHERE scraped_at < {cutoff}")
    log(f"  -> Applied {RETENTION_DAYS}-day rolling retention policy across all tables.")


if df.empty:
    log("[3.4] Zero valid new pages extracted in this cycle. Marking URLs seen & exiting.")
    try:
        mark_seen(attempted)
        apply_retention()
        get_con().execute("CHECKPOINT")
        time.sleep(1.5)
        get_con().close()
    except Exception as e:
        log(f"  -> Cleanup warning on empty run: {e}")
    save_and_shutdown_session()

In [ ]:
# ----------------------------- PII HELPER (Keep in DB; scrub on export/sale) --
log(f"[4.1] Preparing PII helper (keeping emails & phones in DB; scrubbing only at export/sale) across {len(df)} pages...")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_RE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)")
ARABIC_CHAR_RE = re.compile(r"[\u0600-\u06FF]")
CJK_CHAR_RE = re.compile(r"[\u4E00-\u9FFF]")


def scrub_pii_for_sale(text: str) -> str:
    """Use this helper when exporting/selling data so raw emails & phone numbers stay in MotherDuck."""
    if not text:
        return text
    text = EMAIL_RE.sub("[EMAIL]", text)
    text = PHONE_RE.sub("[PHONE]", text)
    return text


email_hits = sum(len(EMAIL_RE.findall(t or "")) for t in df["raw_text"])
phone_hits = sum(len(PHONE_RE.findall(t or "")) for t in df["raw_text"])
log(f"  -> Detected {email_hits} emails and {phone_hits} phone numbers (kept intact in DB; use scrub_pii_for_sale() before selling/exporting).")

# ----------------------------- LLM QUALITY GATE -----------------------------
import gc
log("[4.2] Clearing residual GPU memory before loading LLM...")
if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    alloc_mb = torch.cuda.memory_allocated(0) / (1024 ** 2)
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / (1024 ** 3)
    log(f"  -> GPU 0 pre-load state: allocated={alloc_mb:.1f} MB | free={free_gb:.2f} GB")

t_load = time.time()
log(f"[4.3] Loading {MODEL_ID} (FP16) onto GPU 0 from {CACHED_MODEL_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(CACHED_MODEL_DIR, padding_side="left", local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CACHED_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    local_files_only=True,
)
model.eval()
vram_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
log(f"  -> Model loaded on {model.device} in {time.time() - t_load:.1f}s | GPU 0 VRAM allocated: {vram_gb:.2f} GB")


def build_prompt(niche: str, text: str) -> str:
    user = f"Niche: {niche}\nSchema for extracted_data: {NICHE_SCHEMA[niche]}\n\nTEXT:\n{text}"
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


NULL_STRINGS = {
    "unknown", "n/a", "na", "none", "null", "not specified",
    "not specified in the text", "not mentioned", "not provided", "not available", "",
}


def clean_extracted_dict(d: dict) -> dict:
    """Normalize placeholder null strings into real JSON None."""
    cleaned = {}
    for k, v in d.items():
        if isinstance(v, str) and v.strip().lower() in NULL_STRINGS:
            cleaned[k] = None
        elif isinstance(v, list):
            cleaned[k] = [x for x in v if not (isinstance(x, str) and x.strip().lower() in NULL_STRINGS)]
            if not cleaned[k]:
                cleaned[k] = None
        else:
            cleaned[k] = v
    return cleaned


OFF_REGION_LOC_TERMS = (
    "spokane", "seattle", "san francisco", "philadelphia", "washington, d.c",
    "united states", "usa", "seoul", "seongnam", "korea", "london", "manchester",
    "united kingdom", "toronto", "vancouver", "sydney", "melbourne",
)


def parse_reply(niche: str, reply: str):
    """Returns (parsed_obj_or_None, reason_tag)."""
    cleaned_reply = re.sub(r"^```(?:json)?\s*|\s*```$", "", reply.strip(), flags=re.IGNORECASE)
    s, e = cleaned_reply.find("{"), cleaned_reply.rfind("}") + 1
    if s == -1 or e <= s:
        return None, "invalid_json_bounds"
    try:
        obj = json.loads(cleaned_reply[s:e])
    except Exception:
        return None, "json_decode_error"

    if obj.get("is_sellable") is not True:
        return None, f"not_sellable(score={obj.get('value_score', '?')})"

    try:
        score = int(obj.get("value_score", 0))
    except Exception:
        score = 0
    if score < MIN_VALUE_SCORE:
        return None, f"low_score({score}<{MIN_VALUE_SCORE})"

    if not isinstance(obj.get("extracted_data"), dict) or not obj["extracted_data"]:
        return None, "empty_schema"

    cleaned = clean_extracted_dict(obj["extracted_data"])

    # Strict output check for arabic_nlp: require substantial Arabic script and reject CJK/empty outputs
    if niche == "arabic_nlp":
        ar_txt = str(cleaned.get("clean_arabic_text") or "").strip()
        ar_chars = len(ARABIC_CHAR_RE.findall(ar_txt))
        cjk_chars = len(CJK_CHAR_RE.findall(ar_txt))
        ar_ratio = ar_chars / max(len(ar_txt), 1)
        if len(ar_txt) < 40 or ar_chars < 25 or ar_ratio < 0.55 or cjk_chars > 0:
            return None, f"non_arabic_text(ar={ar_chars},len={len(ar_txt)},cjk={cjk_chars})"

    non_null_fields = sum(1 for v in cleaned.values() if v not in (None, [], ""))
    if non_null_fields < 2:
        return None, f"insufficient_fields({non_null_fields}<2)"

    # Niche-specific quality gates to stop generic directories and off-region rows
    if niche == "b2b_leads":
        loc_val = str(cleaned.get("location") or "").lower()
        if any(m in loc_val for m in OFF_REGION_LOC_TERMS) or re.search(r"\buk\b", loc_val):
            return None, "off_region_b2b_lead"
        if not cleaned.get("company_name") or not cleaned.get("job_title"):
            return None, "missing_company_or_role"

    if niche == "real_estate":
        price_val = cleaned.get("price")
        loc_val = str(cleaned.get("location") or "").strip()
        urg = str(cleaned.get("urgency_signal") or "").lower()
        # Reject empty search pages & news/editorial commentary URLs
        if any(p in urg for p in ("no properties found", "0 properties", "no listings", "settlement", "eviction")):
            return None, "empty_or_editorial_listing"
        # Require a scalar price with digits (reject null, dicts, lists, percentages, or non-numeric text)
        if not isinstance(price_val, (int, float, str)):
            return None, "invalid_price_type"
        price_str = str(price_val).strip()
        if not price_str or "%" in price_str or not re.search(r"\d", price_str):
            return None, "missing_or_non_numeric_price"
        # Reject macro market volume figures (billions) & generic country-only location
        if re.search(r"\b(billion|bn)\b", price_str, flags=re.IGNORECASE):
            return None, "macro_market_stat_not_listing"
        if not loc_val or loc_val.lower() in {"uae", "united arab emirates", "null", "none"}:
            return None, "missing_specific_uae_location"

    if niche == "ecommerce":
        p_val = str(cleaned.get("price") or "").strip()
        if not cleaned.get("product_name") or not p_val or not re.search(r"\d", p_val):
            return None, "missing_product_or_numeric_price"
        # Reject leaked real-estate prices (e.g. AED 1,850,000) unless industrial/machinery
        num_match = re.search(r"([\d,]+(?:\.\d+)?)", p_val)
        if num_match:
            try:
                numeric_price = float(num_match.group(1).replace(",", ""))
                prod_lower = str(cleaned.get("product_name") or "").lower()
                if numeric_price > 100000 and not any(k in prod_lower for k in ("server", "machinery", "generator", "vehicle", "fleet")):
                    return None, f"unrealistic_ecommerce_price({numeric_price})"
            except Exception:
                pass

    if niche == "tenders":
        if not cleaned.get("tender_title") or (not cleaned.get("issuing_org") and not cleaned.get("submission_deadline")):
            return None, "incomplete_tender"
        # Reject expired tenders when an explicit past year/date is extracted
        deadline_str = str(cleaned.get("submission_deadline") or "").strip()
        if deadline_str:
            parsed_dt = pd.to_datetime(deadline_str, errors="coerce", dayfirst=True)
            if pd.notna(parsed_dt):
                today_naive = datetime.now(timezone.utc).replace(tzinfo=None, hour=0, minute=0, second=0, microsecond=0)
                if parsed_dt.to_pydatetime() < today_naive:
                    return None, f"expired_tender({deadline_str})"

    obj["extracted_data"] = cleaned
    return obj, f"KEEP(score={score}, fields={non_null_fields})"


log(f"[5.1] Formatting & length-sorting {len(df)} prompts for batched inference (batch_size={LLM_BATCH_SIZE})...")
df = df.assign(prompt=[build_prompt(n, t) for n, t in zip(df["niche"], df["raw_text"])])
df = df.assign(plen=df["prompt"].str.len()).sort_values("plen")
rows = list(df.itertuples())
sellable = []
discard_reasons = {}
total_batches = (len(rows) + LLM_BATCH_SIZE - 1) // LLM_BATCH_SIZE
t_inf = time.time()

for b_idx, i in enumerate(range(0, len(rows), LLM_BATCH_SIZE), start=1):
    t_b = time.time()
    batch = rows[i:i + LLM_BATCH_SIZE]
    inputs = tokenizer(
        [r.prompt for r in batch],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1664,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    for r, reply in zip(batch, decoded):
        obj, reason = parse_reply(r.niche, reply.strip())
        short_url = (r.source_url[:62] + "...") if len(r.source_url) > 65 else r.source_url
        if obj:
            sellable.append({
                "niche": r.niche,
                "source_url": r.source_url,
                "scraped_at": r.scraped_at,
                "value_score": int(obj["value_score"]),
                "cleaned_json": json.dumps(obj["extracted_data"], ensure_ascii=False),
            })
            log(f"    + [{r.niche}] {reason} | {short_url}")
        else:
            tag = reason.split("(")[0]
            discard_reasons[tag] = discard_reasons.get(tag, 0) + 1
            log(f"    - [{r.niche}] DROP:{reason} | {short_url}")

    del inputs, out, new_tokens
    torch.cuda.empty_cache()
    peak_gb = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
    done = min(i + LLM_BATCH_SIZE, len(rows))
    log(f"  -> Batch [{b_idx}/{total_batches}] ({done}/{len(rows)} pages) done in {time.time() - t_b:.1f}s | kept={len(sellable)} | peak_vram={peak_gb:.2f} GB")

log(f"[5.2] Quality gate finished in {time.time() - t_inf:.1f}s: {len(rows)} evaluated -> {len(sellable)} KEPT, {len(rows) - len(sellable)} DISCARDED.")
if discard_reasons:
    log(f"  -> Discard breakdown: {discard_reasons}")

In [ ]:
# ----------------------------- SAVE -----------------------------------------
try:
    log("[6.1] Marking attempted URLs in MotherDuck 'seen_urls'...")
    db = get_con()
    mark_seen(attempted)

    if sellable:
        log(f"[6.2] Inserting {len(sellable)} sellable records into MotherDuck niche tables...")
        clean_df = pd.DataFrame(sellable).drop_duplicates(["niche", "source_url"]).copy()
        clean_df["scraped_at"] = pd.to_datetime(clean_df["scraped_at"], utc=True)
        clean_df["value_score"] = clean_df["value_score"].astype("int64")

        for niche, group in clean_df.groupby("niche"):
            existing_niche_urls = {
                r[0] for r in db.execute(f"SELECT source_url FROM {niche}").fetchall()
            }
            group = group[~group["source_url"].isin(existing_niche_urls)].copy()
            if group.empty:
                log(f"  -> 'minedb.{niche}': 0 new rows (all already existed).")
                continue
            try:
                db.unregister("group_view")
            except Exception:
                pass
            db.register("group_view", group)
            db.execute(f"""
                INSERT INTO {niche} (niche, source_url, scraped_at, value_score, cleaned_json)
                SELECT
                    CAST(niche AS VARCHAR),
                    CAST(source_url AS VARCHAR),
                    CAST(scraped_at AS TIMESTAMPTZ),
                    CAST(value_score AS INTEGER),
                    CAST(cleaned_json AS VARCHAR)
                FROM group_view
            """)
            try:
                db.unregister("group_view")
            except Exception:
                pass
            new_total = db.execute(f"SELECT COUNT(*) FROM {niche}").fetchone()[0]
            log(f"  -> 'minedb.{niche}': inserted +{len(group)} new rows (table total now {new_total}).")
        print("\n--- SAMPLE EXTRACTED RECORDS ---", flush=True)
        print(clean_df[["niche", "value_score", "cleaned_json"]].head(8).to_string(index=False), flush=True)
    else:
        log(f"[6.2] No rows met value_score >= {MIN_VALUE_SCORE}. 'minedb' tables remain clean.")

    log("[6.3] Running rolling retention cleanup & flushing MotherDuck connection...")
    apply_retention()
    try:
        db.execute("CHECKPOINT")
        time.sleep(1.5)
        db.close()
    except Exception:
        pass
    log("=== PIPELINE COMPLETED SUCCESSFULLY ===")
except Exception as save_err:
    log(f"[6.X] ERROR during save/retention stage: {save_err!r}")
    traceback.print_exc()
    sys.stdout.flush()
    sys.stderr.flush()
    raise
finally:
    save_and_shutdown_session()

In [ ]:
# ----------------------------- EXPORT / SALE SANITIZER -------------------
import json
import re
import duckdb

EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_RE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)")


def scrub_pii_for_sale(text: str) -> str:
    if not text:
        return text
    text = EMAIL_RE.sub("[EMAIL]", text)
    text = PHONE_RE.sub("[PHONE]", text)
    return text


def export_for_sale(niches=None, out_path="sale_export.jsonl"):
    """Reads from MotherDuck (where raw emails/phones remain stored) and writes a PII-scrubbed file for sale."""
    target_niches = niches or list(NICHE_QUERIES.keys())
    con_ro = duckdb.connect("md:minedb")
    exported = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for n in target_niches:
            rows = con_ro.execute(
                f"SELECT niche, source_url, scraped_at, value_score, cleaned_json FROM {n}"
            ).fetchall()
            for niche, src, scraped, score, cjson in rows:
                record = {
                    "niche": niche,
                    "source_url": src,
                    "scraped_at": str(scraped),
                    "value_score": score,
                    "cleaned_json": scrub_pii_for_sale(cjson),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                exported += 1
    con_ro.close()
    print(f"Exported {exported} PII-scrubbed records to {out_path}")
